In [0]:
%run ../init_framework

In [0]:
# --- 1. Initialize the CDF Read Stream for Defaulters ---
df_bronze_defaulters = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) 
    .table(DEFAULTERS_BRONZE))

In [0]:
df_with_metadata = apply_silver_metadata(df_bronze_defaulters)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
from pyspark.sql.functions import col

# 1. Cast to float first to avoid losing decimal strings to NULL
# 2. Cast to int to perform the "floor" truncation
# 3. Replace all NULLs (original nulls + invalid strings) with 0

df_delinq2yrs_fixed = df_distinct.withColumn(
    "delinq_2yrs", col("delinq_2yrs").cast("float").cast("int")
).fillna(0, subset=["delinq_2yrs"])

In [0]:
from pyspark.sql.functions import col

df_defaulters_delinq = (df_delinq2yrs_fixed
    .select(
        col("member_id"),
        col("delinq_2yrs"),
        col("delinq_amnt"),
        col("mths_since_last_delinq").cast("int"),
        # Metadata Columns
        col("_ingested_at"),
        col("_source_file"),
        col("_bronze_version"),
        col("_updated_at")
    )
    .filter((col("delinq_2yrs") > 0) | (col("mths_since_last_delinq") > 0))
)

In [0]:
# final dedup on PK to prevent merge conflicts in the foreachBatch
df_defaulters_delinq_final = df_defaulters_delinq.dropDuplicates(["member_id"])
 
def upsert_defaulters_delinq_to_silver(microBatchDF, batchId):
    """
    stateless MERGE from bronze CDF to silver defaulters (delinquency).
    """
    spark_session = microBatchDF.sparkSession

    # register view for spark sql engine
    microBatchDF.createOrReplaceTempView("defaulters_delinq_batch_updates")

    # explicit sql mapping. joined strictly on member_id.
    merge_query = f"""
    MERGE INTO {DEFAULTERS_DELINQ_SILVER} AS target
        USING defaulters_delinq_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.delinq_2yrs = source.delinq_2yrs,
            target.delinq_amnt = source.delinq_amnt,
            target.mths_since_last_delinq = source.mths_since_last_delinq,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, delinq_2yrs, delinq_amnt, mths_since_last_delinq,
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.member_id, source.delinq_2yrs, source.delinq_amnt, source.mths_since_last_delinq,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

# optimize shuffle partitions based on our cluster size (8 cores)
spark.conf.set("spark.sql.shuffle.partitions", "16")
 
# --- start streaming pipeline ---
query_defaulters_delinq = (
    df_defaulters_delinq_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_DELINQ)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_delinq_to_silver)
    .start()
)

# block cluster exit until this stream finishes
query_defaulters_delinq.awaitTermination()

In [0]:
%sql
select count(1) from lending_risk.silver.defaulters_delinq;
-- 737137
-- select * from lending_risk.silver.defaulters_delinq limit 3;

In [0]:
from pyspark.sql.functions import col

# Direct DataFrame API implementation for Enquiries with Metadata
df_defaulters_inquiry = (df_delinq2yrs_fixed
    .select(
        col("member_id"), 
        col("pub_rec"), 
        col("pub_rec_bankruptcies"), 
        col("inq_last_6mths"),
        # Metadata Columns included in projection
        col("_ingested_at"),
        col("_source_file"),
        col("_bronze_version"),
        col("_updated_at")
    )
    .filter(
        (col("pub_rec") > 0.0) | 
        (col("pub_rec_bankruptcies") > 0.0) | 
        (col("inq_last_6mths") > 0.0)
    )
)

In [0]:
# Final deduplication on the primary key to prevent MERGE conflicts
df_defaulters_inquiry_final = df_defaulters_inquiry.dropDuplicates(["member_id"])

# COMMAND ----------
def upsert_defaulters_inquiry_to_silver(microBatchDF, batchId):
    """
    Stateless MERGE from Bronze CDF into Silver Defaulters Inquiry using explicit SQL mapping.
    """
    spark_session = microBatchDF.sparkSession

    # Register the incoming micro-batch as a Temp View
    microBatchDF.createOrReplaceTempView("defaulters_inquiry_batch_updates")

    # Execute explicit SQL MERGE. 
    # Joined strictly on member_id.
    merge_query = f"""
    MERGE INTO {DEFAULTERS_INQUIRY_SILVER} AS target
        USING defaulters_inquiry_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.pub_rec = source.pub_rec,
            target.pub_rec_bankruptcies = source.pub_rec_bankruptcies,
            target.inq_last_6mths = source.inq_last_6mths,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths,
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.member_id, source.pub_rec, source.pub_rec_bankruptcies, source.inq_last_6mths,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)

# Configure shuffle partitions for local cluster core count (8 cores -> 16 partitions)
spark.conf.set("spark.sql.shuffle.partitions", "16")
 
# --- Start Streaming Pipeline ---
query_defaulters_inquiry = (
    df_defaulters_inquiry_final.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_INQUIRY)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_inquiry_to_silver)
    .start()
)

# Block cluster exit until this stream finishes
query_defaulters_inquiry.awaitTermination()

In [0]:
%sql
select * from lending_risk.silver.defaulters_inquiry limit 3;
-- select count(1) from lending_risk.silver.defaulters_inquiry;
-- 713068